In [ ]:
from aatm.data_models import ExtractionTask
from aatm.extractors import GeminiExtractor


text = """
Nome do paciente: João Almeida

Token da Receita: SDF234G

Data da Receita
15/06/2025
Emitida

Receita

Alivium 600 mg, Cápsula mole
Tomar 1 cápsula via oral a cada 6 horas se dor não aliviar com paracetamol.
1x Caixa com 10 Cápsulas

Paracetamol 750 mg, Comprimido
Tomar 1 comprimido via oral a cada 6 horas se dor ou febre.
1x Caixa com 10 Comprimidos
"""

prompt_template = """Your task is to extract data related to drug exposure in the OMOP common data model. Only extract information that is present in the context given. Do not use your world knowledge to fill in information that is not explicitly stated in the context provided. Return a JSON string following the required format.

# Text
"{text}"""

data_model = "drug_exposure_list" # or ListOfDrugExposureExtractionModel object

# You may define one or multiple tasks
task = ExtractionTask(
    prompt_template=prompt_template,
    texts=[text],
    data_model=data_model
)

extractor = GeminiExtractor(
    model_id="gemini-2.5-flash"
)

In [2]:
# extractors expect a list of tasks
results = extractor([task])

In [3]:
# Let's look at the first result of the first task
results[0][0].model_dump()

{'records': [{'drug_source_value': 'Alivium 600 mg, Cápsula mole',
   'drug_source_concept_id': None,
   'drug_exposure_start_date': datetime.date(2025, 6, 15),
   'drug_exposure_start_datetime': None,
   'drug_exposure_end_date': None,
   'drug_exposure_end_datetime': None,
   'verbatim_end_date': None,
   'stop_reason': None,
   'refills': None,
   'quantity': 10.0,
   'days_supply': None,
   'sig': 'Tomar 1 cápsula via oral a cada 6 horas se dor não aliviar com paracetamol.',
   'lot_number': None,
   'route_source_value': 'oral',
   'dose_unit_source_value': 'mg'},
  {'drug_source_value': 'Paracetamol 750 mg, Comprimido',
   'drug_source_concept_id': None,
   'drug_exposure_start_date': datetime.date(2025, 6, 15),
   'drug_exposure_start_datetime': None,
   'drug_exposure_end_date': None,
   'drug_exposure_end_datetime': None,
   'verbatim_end_date': None,
   'stop_reason': None,
   'refills': None,
   'quantity': 10.0,
   'days_supply': None,
   'sig': 'Tomar 1 comprimido via oral

In [9]:
# DrugExposure model contains the remaining OMOP fields that are required for a complete representation of a drug_exposure record, but are not expected to be explicit in clinical documents

from aatm.omop.drug_exposure import DrugExposure

extracted_data = results[0][0].records[0].model_dump()

DrugExposure(
    drug_exposure_id=123,
    person_id=123,
    drug_concept_id=123,
    drug_type_concept_id=123,
    provider_id=123,
    visit_occurrence_id=123,
    visit_detail_id=123,
    route_concept_id=123,
    **extracted_data
).model_dump()

# You can combine an extractor pipeline with a terminology mapping pipeline to find relevant concept ids if needed. 

{'drug_source_value': 'Alivium 600 mg, Cápsula mole',
 'drug_source_concept_id': None,
 'drug_exposure_start_date': datetime.date(2025, 6, 15),
 'drug_exposure_start_datetime': None,
 'drug_exposure_end_date': None,
 'drug_exposure_end_datetime': None,
 'verbatim_end_date': None,
 'stop_reason': None,
 'refills': None,
 'quantity': 10.0,
 'days_supply': None,
 'sig': 'Tomar 1 cápsula via oral a cada 6 horas se dor não aliviar com paracetamol.',
 'lot_number': None,
 'route_source_value': 'oral',
 'dose_unit_source_value': 'mg',
 'drug_exposure_id': 123,
 'person_id': 123,
 'drug_concept_id': 123,
 'drug_type_concept_id': 123,
 'provider_id': 123,
 'visit_occurrence_id': 123,
 'visit_detail_id': 123,
 'route_concept_id': 123}